In [1]:
import torch
from torchinfo import summary


from improved_diffusion.models.vae import AbstractVAE

In [2]:
DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)

print(f"Using device: {DEVICE}")

DIMS = 2

# 4 latent channels: forces the model to use each channel efficiently.
# For binary fiber microstructure, 4 channels cover density, orientation,
# scale, and residual — more than enough at 32×32 spatial resolution.
vae = AbstractVAE(
    in_channels=1,
    latent_channels=4,
    base_channels=32,
    channel_mult=(1, 2, 4, 4, 4), # 32, 64, 128, 128, 128 channels at each resolution level
    dims=DIMS,
    attn_ds=( 8, 16), # attention at 1/4, 1/8, and 1/16 resolution
    attn_heads=1,
    spatial_latent=True,
).to(DEVICE)

Using device: mps


In [3]:

size = (1, 1) + (128,) * DIMS


summary(vae,input_size=(size))

Layer (type:depth-idx)                   Output Shape              Param #
AbstractVAE                              [1, 1, 128, 128]          --
├─Sequential: 1-1                        [1, 128, 8, 8]            --
│    └─Conv2d: 2-1                       [1, 32, 128, 128]         320
│    └─ResBlock: 2-2                     [1, 32, 128, 128]         --
│    │    └─Sequential: 3-1              [1, 32, 128, 128]         18,624
│    └─Downsample: 2-3                   [1, 64, 64, 64]           --
│    │    └─Conv2d: 3-2                  [1, 64, 64, 64]           32,832
│    └─ResBlock: 2-4                     [1, 64, 64, 64]           --
│    │    └─Sequential: 3-3              [1, 64, 64, 64]           74,112
│    └─Downsample: 2-5                   [1, 128, 32, 32]          --
│    │    └─Conv2d: 3-4                  [1, 128, 32, 32]          131,200
│    └─ResBlock: 2-6                     [1, 128, 32, 32]          --
│    │    └─Sequential: 3-5              [1, 128, 32, 32]          